```{contents}
```
## Learning Rate Scheduling


The **learning rate (LR)** controls the step size taken by the optimizer when updating model parameters.
A fixed LR is rarely optimal for the entire training process:

* **Large LR** → fast progress but unstable, may overshoot minima
* **Small LR** → stable but slow, may get stuck in poor local minima

**Learning Rate Scheduling** dynamically adjusts the LR over training to balance:

> **exploration early** and **fine convergence later**

Intuitively:

* Early training needs **large steps** to escape bad regions.
* Late training needs **small steps** to finely settle into a good minimum.

---

### Optimization View

Parameter update:
$$
\theta_{t+1} = \theta_t - \eta_t \nabla L(\theta_t)
$$

where:

* $\eta_t$ is the learning rate at step $t$
* Scheduling defines how $\eta_t$ changes with time

---

### Training Workflow With Scheduling

1. Initialize model and optimizer with base LR
2. Define LR scheduler
3. For each epoch or step:

   * Forward pass
   * Backward pass
   * Optimizer step
   * Scheduler step (updates LR)

---

### Why Scheduling Improves Training

| Problem          | How Scheduling Helps               |
| ---------------- | ---------------------------------- |
| Slow convergence | High LR early accelerates learning |
| Divergence       | LR decay stabilizes updates        |
| Poor minima      | Warm restarts help escape          |
| Overfitting late | Lower LR acts as regularizer       |

---

### Major Scheduling Strategies

#### Step Decay

Reduce LR by factor at fixed intervals.

$$
\eta_t = \eta_0 \cdot \gamma^{\lfloor t / s \rfloor}
$$

Use when training plateaus periodically.

#### Exponential Decay

Smooth decay:

$$
\eta_t = \eta_0 \cdot e^{-kt}
$$

Stable and simple.

#### Cosine Annealing

Gradually decays LR following cosine curve:

$$
\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})(1 + \cos(\pi t / T))
$$

Encourages better minima.

#### Reduce On Plateau

Decrease LR when validation loss stops improving.

Data-driven and adaptive.

#### Warmup

Start with very small LR and increase to base LR:

Prevents unstable early training, especially for transformers.

---

### PyTorch Demonstration

```python
import torch
from torch import nn
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, ReduceLROnPlateau

model = nn.Linear(100, 10)
optimizer = Adam(model.parameters(), lr=1e-3)
```

#### StepLR Example

```python
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)

for epoch in range(50):
    train_one_epoch(model)
    optimizer.step()
    scheduler.step()
```

#### Cosine Annealing

```python
scheduler = CosineAnnealingLR(optimizer, T_max=50)

for epoch in range(50):
    train_one_epoch(model)
    optimizer.step()
    scheduler.step()
```

#### Reduce on Plateau

```python
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3)

for epoch in range(50):
    train_loss, val_loss = train_and_validate(model)
    optimizer.step()
    scheduler.step(val_loss)
```

---

### Warmup Integration

```python
from torch.optim.lr_scheduler import LambdaLR

def warmup_fn(epoch):
    if epoch < 5:
        return epoch / 5
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda=warmup_fn)
```

---

### Remediation and Debugging

| Symptom               | Fix                            |
| --------------------- | ------------------------------ |
| Loss oscillates       | Lower initial LR or add warmup |
| Training stalls       | Increase LR or delay decay     |
| Overfitting late      | Stronger LR decay              |
| Unstable first epochs | Apply warmup                   |

---

### Practical Guidelines

| Scenario        | Recommended Strategy |
| --------------- | -------------------- |
| CNN training    | StepLR or Cosine     |
| Transformers    | Warmup + Cosine      |
| Unknown dataset | ReduceOnPlateau      |
| Long training   | Cosine with restarts |

---

### Summary

Learning rate scheduling transforms training from a rigid process into a **controlled optimization trajectory**.
Proper scheduling dramatically improves convergence speed, stability, and final performance while reducing the need for manual tuning.